In [1]:
import sys

import numpy
import pandas
import sklearn

print(sys.version.split()[0], numpy.__version__, pandas.__version__, sklearn.__version__)

3.12.10 2.0.2 2.3.3 1.6.1


# Task 2 — Decision Trees
Gumar Nurtore

SEED = 3242 (last four digits of my student ID).

In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

SEED = 3242

data = pd.read_csv("data/heart_disease.csv")

feature_names = [c for c in data.columns if c != "disease"]

X = data[feature_names].to_numpy()
y = data["disease"].to_numpy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    stratify=y,
    random_state=SEED
)

print(data.shape, round(y.mean(), 3), X_train.shape, X_test.shape)

(297, 14) 0.461 (207, 13) (90, 13)


# Part 1

## Parent entropy

There are 12 patients: 6 with disease and 6 without disease.

p1 = 6/12 = 0.5
p0 = 6/12 = 0.5

H = -(6/12) * log2(6/12) - (6/12) * log2(6/12)
  = -0.5 * (-1) - 0.5 * (-1)
  = 1 bit.

Both classes have the same number of patients, so the entropy is 1.

## Information gain: exercise_angina

Parent entropy = 1 bit.

exercise_angina = 1: 4 patients, all with disease.
H1 = 0.

exercise_angina = 0: 8 patients, 2 with disease and 6 without.
H0 = -(2/8) * log2(2/8) - (6/8) * log2(6/8)
   ≈ 0.811278.

Weighted entropy = (4/12) * 0 + (8/12) * 0.811278
                 ≈ 0.540852.

Information gain = 1 - 0.540852
                 ≈ 0.459148 bits.

## Information gain: other features

### age55
age55 = 1: 5 with disease, 1 without.
age55 = 0: 1 with disease, 5 without.

Both groups have the same entropy:
H = -(5/6) * log2(5/6) - (1/6) * log2(1/6)
  ≈ 0.650022.

IG = 1 - (6/12) * 0.650022 - (6/12) * 0.650022
   ≈ 0.349978.

### male
Both groups have 3 patients with disease and 3 without.

H = -(3/6) * log2(3/6) - (3/6) * log2(3/6)
  = 1.

IG = 1 - (6/12) * 1 - (6/12) * 1 = 0.

### high_bp
high_bp = 1: 4 with disease, 2 without.
high_bp = 0: 2 with disease, 4 without.

Both groups have the same entropy:
H = -(4/6) * log2(4/6) - (2/6) * log2(2/6)
  ≈ 0.918296.

IG = 1 - (6/12) * 0.918296 - (6/12) * 0.918296
   ≈ 0.081704.

## Best root split

| Feature | Information gain |
|---------|------------------|
| exercise_angina | 0.459148 |
| age55 | 0.349978 |
| high_bp | 0.081704 |
| male | 0.000000 |

I choose exercise_angina for the root because it has the highest
information gain.

# Part 2

## Entropy function

In [3]:
def entropy(y):
    """Calculate entropy in bits for a list of class labels."""
    y = np.asarray(y)

    if len(y) == 0:
        return 0.0

    labels, counts = np.unique(y, return_counts=True)
    probabilities = counts / len(y)

    return float(-np.sum(probabilities * np.log2(probabilities)))

In [5]:
# Проверка
print("All the same:", entropy([0, 0, 0, 0]))
print("Half and half:", entropy([0, 0, 1, 1]))
print("Six zeros, four ones:", entropy([0, 0, 0, 0, 0, 0, 1, 1, 1, 1]))

All the same: -0.0
Half and half: 1.0
Six zeros, four ones: 0.9709505944546686


In [6]:
def information_gain(y, left):
    """Calculate information gain for a split given by a boolean mask."""
    y = np.asarray(y)
    left = np.asarray(left, dtype=bool)

    if len(y) == 0:
        return 0.0

    y_left = y[left]
    y_right = y[~left]

    if len(y_left) == 0 or len(y_right) == 0:
        return 0.0

    weight_left = len(y_left) / len(y)
    weight_right = len(y_right) / len(y)

    entropy_after = (
        weight_left * entropy(y_left)
        + weight_right * entropy(y_right)
    )

    return entropy(y) - entropy_after

In [7]:
# проверка
answers = np.array([1, 1, 0, 0])

good_split = np.array([True, True, False, False])
bad_split = np.array([True, False, True, False])

print("Good split:", information_gain(answers, good_split))
print("Bad split:", information_gain(answers, bad_split))

Good split: 1.0
Bad split: 0.0


In [8]:
def best_split(X, y, min_samples_leaf=1):
    """Return (feature index, threshold, gain), or None if no valid split."""
    X = np.asarray(X)
    y = np.asarray(y)

    best = None
    best_gain = -np.inf

    for feature in range(X.shape[1]):
        values = np.unique(X[:, feature])
        thresholds = (values[:-1] + values[1:]) / 2

        for threshold in thresholds:
            left = X[:, feature] <= threshold

            n_left = np.sum(left)
            n_right = len(y) - n_left

            if n_left < min_samples_leaf or n_right < min_samples_leaf:
                continue

            gain = information_gain(y, left)

            if gain > best_gain:
                best_gain = gain
                best = (feature, float(threshold), float(gain))

    return best

In [18]:
X_example = np.array([[20], [50], [50], [60]])
y_example = np.array([1, 0, 1, 0])

print(best_split(X_example, y_example))

(0, 35.0, 0.31127812445913283)


In [19]:
def build_tree(X, y, max_depth=None, min_samples_leaf=1, depth=0):
    """Build a decision tree using entropy and information gain."""
    X = np.asarray(X)
    y = np.asarray(y)

    labels, counts = np.unique(y, return_counts=True)
    prediction = labels[np.argmax(counts)]

    # Stop if the group is pure or the depth limit is reached.
    if len(labels) == 1 or (max_depth is not None and depth >= max_depth):
        return {"prediction": prediction}

    split = best_split(X, y, min_samples_leaf=min_samples_leaf)

    # Stop if no valid split exists.
    if split is None:
        return {"prediction": prediction}

    feature, threshold, gain = split
    left = X[:, feature] <= threshold

    return {
        "feature": feature,
        "threshold": threshold,
        "gain": gain,
        "prediction": prediction,
        "left": build_tree(
            X[left], y[left], max_depth, min_samples_leaf, depth + 1
        ),
        "right": build_tree(
            X[~left], y[~left], max_depth, min_samples_leaf, depth + 1
        ),
    }

In [22]:
# проверка
small_tree = build_tree(X_example, y_example, max_depth=1)
print(small_tree)

{'feature': 0, 'threshold': 35.0, 'gain': 0.31127812445913283, 'prediction': np.int64(0), 'left': {'prediction': np.int64(1)}, 'right': {'prediction': np.int64(0)}}


In [23]:
def predict_tree(tree, X):
    """Predict class labels for rows in X using our tree."""
    X = np.asarray(X)
    predictions = []

    for patient in X:
        node = tree

        while "feature" in node:
            feature = node["feature"]
            threshold = node["threshold"]

            if patient[feature] <= threshold:
                node = node["left"]
            else:
                node = node["right"]

        predictions.append(node["prediction"])

    return np.array(predictions)

In [24]:
new_patients = np.array([[20], [35], [45], [70]])
print(predict_tree(small_tree, new_patients))

[1 1 0 0]


## Checking the functions on the toy data

In [26]:
toy_data = pd.DataFrame({
    "age55":            [1, 1, 1, 1, 1, 0, 1, 0, 0, 0, 0, 0],
    "exercise_angina":  [1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0],
    "male":             [1, 1, 1, 0, 0, 0, 1, 1, 1, 0, 0, 0],
    "high_bp":          [1, 1, 0, 1, 1, 0, 0, 0, 0, 1, 1, 0],
    "disease":          [1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0]
})

toy_data

,age55,exercise_angina,male,high_bp,disease
0,1,1,1,1,1
1,1,1,1,1,1
2,1,1,1,0,1
3,1,1,0,1,1
4,1,0,0,1,1
5,0,0,0,0,1
6,1,0,1,0,0
7,0,0,1,0,0
8,0,0,1,0,0
9,0,0,0,1,0


In [27]:
toy_features = ["age55", "exercise_angina", "male", "high_bp"]

X_toy = toy_data[toy_features].to_numpy()
y_toy = toy_data["disease"].to_numpy()

print("Parent entropy:", entropy(y_toy))

for feature_index, feature_name in enumerate(toy_features):
    left = X_toy[:, feature_index] <= 0.5
    gain = information_gain(y_toy, left)
    print(feature_name, ":", round(gain, 6))

Parent entropy: 1.0
age55 : 0.349978
exercise_angina : 0.459148
male : 0.0
high_bp : 0.081704


In [28]:
toy_best = best_split(X_toy, y_toy)

print("Best split:", toy_best)
print("Feature name:", toy_features[toy_best[0]])

Best split: (1, 0.5, 0.4591479170272448)
Feature name: exercise_angina


In [29]:
# Patients without exercise angina
mask = X_toy[:, 1] == 0

X_branch = X_toy[mask]
y_branch = y_toy[mask]

print("Patients:", len(y_branch))
print("Entropy:", entropy(y_branch))

for i, name in enumerate(toy_features):
    if name == "exercise_angina":
        continue

    left = X_branch[:, i] <= 0.5
    gain = information_gain(y_branch, left)
    print(name, ":", round(gain, 6))

result = best_split(X_branch, y_branch)
print("Best feature:", toy_features[result[0]])

Patients: 8
Entropy: 0.8112781244591328
age55 : 0.073761
male : 0.204434
high_bp : 0.015712
Best feature: male


In [30]:
from sklearn.tree import DecisionTreeClassifier

# Our algorithm
my_split = best_split(X_train, y_train)

print("Our feature:", feature_names[my_split[0]])
print("Our threshold:", my_split[1])
print("Our gain:", my_split[2])

# Scikit-learn algorithm
sk_tree = DecisionTreeClassifier(
    criterion="entropy",
    max_depth=1,
    random_state=SEED
)

sk_tree.fit(X_train, y_train)

print("Sklearn feature:", feature_names[sk_tree.tree_.feature[0]])
print("Sklearn threshold:", sk_tree.tree_.threshold[0])

Our feature: thal
Our threshold: 4.5
Our gain: 0.2227690847093109
Sklearn feature: thal
Sklearn threshold: 4.5


In [31]:
for depth in [1, 2, 3, 4, 5, None]:

    # Our tree
    my_tree = build_tree(X_train, y_train, max_depth=depth)
    my_pred = predict_tree(my_tree, X_test)

    # Sklearn tree
    sk_tree = DecisionTreeClassifier(
        criterion="entropy",
        max_depth=depth,
        random_state=SEED
    )
    sk_tree.fit(X_train, y_train)
    sk_pred = sk_tree.predict(X_test)

    # Compare predictions
    agreement = np.mean(my_pred == sk_pred)

    print("Depth:", depth, "Agreement:", round(agreement, 3))

Depth: 1 Agreement: 1.0
Depth: 2 Agreement: 1.0
Depth: 3 Agreement: 1.0
Depth: 4 Agreement: 1.0
Depth: 5 Agreement: 0.967
Depth: None Agreement: 0.978


### Comparison with scikit-learn

My best_split and scikit-learn selected the same feature
(thal) and the same threshold (4.5).

| Max Depth | Agreement |
|-----------|-----------|
| 1 | 100% |
| 2 | 100% |
| 3 | 100% |
| 4 | 100% |
| 5 | 96.7% |
| None | 97.8% |

Both trees gave the same predictions for depths 1 to 4.

For deeper trees, some predictions were different.
This can happen because the algorithms may choose
different splits when information gains are equal
or very close.